# L. Edge Cases

- **L1**: All same class (degenerate)
- **L2**: Missing values (NaN in input)
- **L3**: Different input scales + normalization
- **L4**: Very small dataset (n<10)

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
from network import Network, NetworkConfig
from optimizer import Adam

## L1: All Same Class

In [ ]:
np.random.seed(42)
config = NetworkConfig(layers=[10, 8, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
net = Network(config)
opt = Adam(learning_rate=0.001)

x = np.random.randn(50, 10)
y = np.zeros((50, 2)); y[:, 0] = 1  # all class 0

try:
    for _ in range(100):
        net.forward(x)
        nw, nb = net.backward(y)
        opt.update(net, nw, nb)
    out = net.forward(x)
    loss = net.loss(y, out)
    preds = np.argmax(out, axis=1)
    print(f'Loss: {loss:.6f}')
    print(f'All predict class 0: {np.all(preds == 0)}')
    print(f'NaN: {np.any(np.isnan(out))}')
    print(f'Status: {"PASS" if not np.any(np.isnan(out)) else "FAIL"}')
except Exception as e:
    print(f'CRASHED: {e}')
    print('Status: FAIL')

## L2: NaN in Input Data

In [ ]:
np.random.seed(42)
config = NetworkConfig(layers=[10, 8, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
net = Network(config)

x = np.random.randn(10, 10)
x[0, 0] = np.nan; x[3, 5] = np.nan
y = np.zeros((10, 2))
y[np.arange(10), np.random.randint(0, 2, 10)] = 1

out = net.forward(x)
has_nan = np.any(np.isnan(out))

print(f'Input has NaN:  True')
print(f'Output has NaN: {has_nan}')
if has_nan:
    print('WARNING: NaN propagates. Consider adding input validation.')
    print('Status: NEEDS FIX')
else:
    print('Status: PASS')

## L3: Different Input Scales + Normalization

In [ ]:
np.random.seed(42)
x = np.column_stack([
    np.random.randn(50) * 1e6,
    np.random.randn(50) * 1e-6,
    np.random.randn(50) * 100,
    np.random.randn(50),
    np.ones(50) * 42,  # constant feature (zero variance)
])

mean = x.mean(axis=0)
std = x.std(axis=0) + 1e-8
x_norm = (x - mean) / std

print('Before normalization:')
print(f'  Means: {x.mean(axis=0).round(2)}')
print(f'  Stds:  {x.std(axis=0).round(2)}')
print('After normalization:')
print(f'  Means: {x_norm.mean(axis=0).round(6)}')
print(f'  Stds:  {x_norm.std(axis=0).round(6)}')

ok = not np.any(np.isnan(x_norm)) and not np.any(np.isinf(x_norm))

config = NetworkConfig(layers=[5, 4, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
net = Network(config)
out = net.forward(x_norm)
ok2 = not np.any(np.isnan(out))

print(f'\nNormalization clean: {ok} | Forward clean: {ok2}')
print(f'Status: {"PASS" if ok and ok2 else "FAIL"}')

## L4: Very Small Dataset (n < 10)

In [ ]:
for n in [1, 2, 3, 5, 9]:
    np.random.seed(42)
    config = NetworkConfig(layers=[5, 4, 2], activation='relu',
        loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
    net = Network(config)
    opt = Adam(learning_rate=0.001)
    x = np.random.randn(n, 5)
    y = np.zeros((n, 2))
    y[np.arange(n), np.random.randint(0, 2, n)] = 1

    try:
        for _ in range(50):
            net.forward(x)
            nw, nb = net.backward(y)
            opt.update(net, nw, nb)
        loss = net.loss(y, net.forward(x))
        ok = not np.isnan(loss)
        print(f'n={n}: loss={loss:.6f} | {"PASS" if ok else "FAIL"}')
    except Exception as e:
        print(f'n={n}: CRASHED - {e} | FAIL')